# 🎤 Maya1 Voice Engine for Eva
**Free voice cloning with 20+ emotions — runs on Kaggle's free GPU (30 hrs/week)**

### Setup (one time):
1. Click **Settings** (right sidebar) → Turn on **Internet** → Turn on **GPU T4 x2**
2. Click **Run All** (top menu)
3. Wait ~3 minutes for model to download
4. Copy the **public URL** from the last cell
5. Paste it in Eva's Admin Dashboard → Settings → Maya1 URL

### Each time you want voice cloning:
- Open this notebook → Click **Run All** → Copy URL → Eva uses it
- Kaggle gives you **30 hours/week free** — way more than Colab
- Notebook stays active as long as the browser tab is open

### Voice Presets:
- Eva Calm, Eva Therapist, Eva Motivator, Eva Lullaby, Eva Companion
- Or type your own: "30-year-old male, Indian accent, warm deep voice"

### Emotion Tags:
Add inline: `Hello <laugh> this is amazing!`
Supported: `<laugh>` `<whisper>` `<sigh>` `<cry>` `<gasp>` `<angry>` `<giggle>` `<chuckle>` and 12+ more

In [ ]:
# Step 1: Install dependencies
!pip install -q torch transformers snac soundfile gradio numpy

In [ ]:
# Step 2: Load Maya1 (3B param voice AI model)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from snac import SNAC
import soundfile as sf
import numpy as np
import tempfile
import gradio as gr

# Maya1 token IDs
CODE_START = 128257
CODE_END = 128258
CODE_OFFSET = 128266
SNAC_MIN = 128266
SNAC_MAX = 156937
FRAME_SIZE = 7
SOH, EOH, SOA, BOS, EOT = 128259, 128260, 128261, 128000, 128009

print('⏳ Loading Maya1 model (3B params)... this takes ~2 min first time')
model = AutoModelForCausalLM.from_pretrained(
    'maya-research/maya1',
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained('maya-research/maya1', trust_remote_code=True)
print(f'✅ Maya1 loaded! ({len(tokenizer)} tokens)')

print('⏳ Loading SNAC audio decoder...')
snac = SNAC.from_pretrained('hubertsiuzdak/snac_24khz').eval().to('cuda')
print('✅ Everything ready! 🎤')

In [ ]:
# Step 3: Voice generation engine

def make_prompt(desc, text):
    s = lambda id: tokenizer.decode([id])
    return s(SOH) + tokenizer.bos_token + f'<description="{desc}"> {text}' + s(EOT) + s(EOH) + s(SOA) + s(CODE_START)

def generate(text, desc):
    prompt = make_prompt(desc, text)
    inputs = tokenizer(prompt, return_tensors='pt')
    inputs = {k: v.to('cuda') for k, v in inputs.items()}
    
    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=2048, min_new_tokens=28,
            temperature=0.4, top_p=0.9, repetition_penalty=1.1,
            do_sample=True, eos_token_id=CODE_END,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    ids = out[0, inputs['input_ids'].shape[1]:].tolist()
    try: ids = ids[:ids.index(CODE_END)]
    except: pass
    
    codes = [t for t in ids if SNAC_MIN <= t <= SNAC_MAX]
    frames = len(codes) // FRAME_SIZE
    codes = codes[:frames * FRAME_SIZE]
    if frames == 0: return None
    
    l1, l2, l3 = [], [], []
    for i in range(frames):
        s = codes[i*7:(i+1)*7]
        l1.append((s[0]-CODE_OFFSET)%4096)
        l2.extend([(s[1]-CODE_OFFSET)%4096, (s[4]-CODE_OFFSET)%4096])
        l3.extend([(s[2]-CODE_OFFSET)%4096, (s[3]-CODE_OFFSET)%4096, (s[5]-CODE_OFFSET)%4096, (s[6]-CODE_OFFSET)%4096])
    
    tensors = [torch.tensor(l, dtype=torch.long, device='cuda').unsqueeze(0) for l in [l1,l2,l3]]
    with torch.inference_mode():
        audio = snac.decoder(snac.quantizer.from_codes(tensors))[0,0].cpu().numpy()
    
    if len(audio) > 2048: audio = audio[2048:]
    f = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    sf.write(f.name, audio, 24000)
    return f.name

print('✅ Voice engine ready!')

In [ ]:
# Step 4: Launch API (Eva connects to the public URL)

PRESETS = {
    'Eva Calm': 'Female, late 20s, Indian accent. Warm, soft, calm. Low pitch, gentle, slow pacing.',
    'Eva Therapist': 'Female, 30s, warm British accent. Empathetic, thoughtful, professional. Medium pitch, measured pacing.',
    'Eva Motivator': 'Female, 30s, American accent. Energetic, powerful, inspiring. Normal pitch, fast pacing.',
    'Eva Companion': 'Female, 20s, American accent. Casual, friendly, warm. Conversational pacing.',
    'Eva Lullaby': 'Female, 30s, soft Indian accent. Dreamy, whisper-like. Low pitch, airy, very slow.',
    'Eva Deep Male': 'Male, 40s, deep British accent. Warm, wise, calm. Low pitch, rich timbre, slow.',
    'Eva Indian Male': 'Male, 30s, Indian accent. Warm, conversational, friendly. Normal pitch.',
    'Custom': '',
}

def speak(text, preset, custom_desc):
    desc = custom_desc.strip() if preset == 'Custom' and custom_desc.strip() else PRESETS.get(preset, PRESETS['Eva Calm'])
    return generate(text, desc)

demo = gr.Interface(
    fn=speak,
    inputs=[
        gr.Textbox(label='Text', placeholder='What should Eva say? Use <laugh> <whisper> <sigh> for emotions', lines=3),
        gr.Dropdown(choices=list(PRESETS.keys()), value='Eva Calm', label='Voice'),
        gr.Textbox(label='Custom Description (only for Custom)', placeholder='e.g., Male, 30s, Indian accent...'),
    ],
    outputs=gr.Audio(label='Generated Speech'),
    title='🎤 Eva × Maya1 Voice Engine',
    description='Free voice AI with 20+ emotions. Powered by Maya1 on Kaggle GPU.',
)

# ⚠️ COPY THE PUBLIC URL BELOW AND PASTE IN EVA ADMIN DASHBOARD
demo.queue()
demo.launch(share=True, show_error=True)